# Aula 5 — Redes Recorrentes: RNNs e LSTMs (PyTorch)

**CCM-109 — Deep Learning · Prof. Ronaldo Prati (UFABC)**

Este notebook acompanha a Aula 5 e cobre:

1. **RNN na mão** — implementação manual para entender a recorrência
2. **Vanishing gradient** — demonstração do problema
3. **LSTM com `nn.LSTM`** — classificação de sentimento no IMDb
4. **GRU** — mesma tarefa com arquitetura simplificada
5. **LSTM Bidirecional** — aproveitando o contexto futuro
6. **Comparativo** — curvas de aprendizado e métricas

## 0. Setup

In [ ]:
# Instalações (execute apenas uma vez no Colab)
# !pip install datasets --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import re, time

# Verificar dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")

## 1. Dataset — IMDb (análise de sentimento)

O IMDb contém 50.000 críticas de filmes rotuladas como positivas (1) ou negativas (0).

In [ ]:
from datasets import load_dataset

ds = load_dataset("stanfordnlp/imdb")
print(ds)
print("\nExemplo:")
print("Texto :", ds["train"][0]["text"][:200], "...")
print("Rótulo:", ds["train"][0]["label"], "(1=positivo, 0=negativo)")

### 1.1 Pré-processamento: tokenização e vocabulário

Vamos construir um vocabulário simples baseado nas palavras mais frequentes.

In [ ]:
def tokenize(text):
    """Tokenização simples: lowercase + split por não-alfanuméricos."""
    return re.findall(r"[a-z]+", text.lower())

# Construir vocabulário com os N tokens mais frequentes
MAX_VOCAB  = 10_000
MAX_LEN    = 200   # comprimento máximo da sequência
PAD_IDX    = 0
UNK_IDX    = 1

counter = Counter()
for ex in ds["train"]:
    counter.update(tokenize(ex["text"]))

# token → índice (0=PAD, 1=UNK, depois os mais frequentes)
vocab = {tok: idx + 2 for idx, (tok, _) in enumerate(counter.most_common(MAX_VOCAB - 2))}
print(f"Vocabulário: {len(vocab) + 2} tokens")
print(f"Top 10 palavras: {list(vocab.items())[:10]}")

In [ ]:
def encode(text, max_len=MAX_LEN):
    """Converte texto em sequência de índices com padding/truncamento."""
    tokens = tokenize(text)[:max_len]
    ids    = [vocab.get(t, UNK_IDX) for t in tokens]
    # Padding à direita
    ids   += [PAD_IDX] * (max_len - len(ids))
    return ids

class IMDbDataset(Dataset):
    def __init__(self, split):
        data = ds[split]
        self.X = torch.tensor([encode(ex["text"]) for ex in data], dtype=torch.long)
        self.y = torch.tensor([ex["label"]        for ex in data], dtype=torch.long)

    def __len__(self):  return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

train_ds = IMDbDataset("train")
test_ds  = IMDbDataset("test")

BATCH = 64
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH)

print(f"Treino: {len(train_ds)} amostras | Teste: {len(test_ds)} amostras")
X_sample, y_sample = next(iter(train_loader))
print(f"Batch X: {X_sample.shape} | y: {y_sample.shape}")

## 2. RNN na Mão

Antes de usar `nn.LSTM`, vamos implementar manualmente uma célula RNN para entender os cálculos:

$$\mathbf{h}_t = \tanh(W_h\,\mathbf{h}_{t-1} + W_x\,\mathbf{x}_t + \mathbf{b})$$

In [ ]:
class ManualRNNCell(nn.Module):
    """Célula RNN implementada manualmente (sem nn.RNN)."""

    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        # Pesos — mesmos parâmetros usados em todos os passos
        self.W_x = nn.Linear(input_size,  hidden_size, bias=False)
        self.W_h = nn.Linear(hidden_size, hidden_size, bias=True)

    def forward(self, x, h_prev):
        # x:      (batch, input_size)
        # h_prev: (batch, hidden_size)
        return torch.tanh(self.W_x(x) + self.W_h(h_prev))

class ManualRNN(nn.Module):
    """RNN completa usando ManualRNNCell."""

    def __init__(self, vocab_size, embed_dim, hidden_size, num_classes):
        super().__init__()
        self.embed  = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.cell   = ManualRNNCell(embed_dim, hidden_size)
        self.fc     = nn.Linear(hidden_size, num_classes)
        self.hidden_size = hidden_size

    def forward(self, x):                      # x: (batch, seq_len)
        emb = self.embed(x)                    # (batch, seq_len, embed_dim)
        batch = x.size(0)
        h = torch.zeros(batch, self.hidden_size, device=x.device)

        for t in range(emb.size(1)):           # percorre cada passo de tempo
            h = self.cell(emb[:, t, :], h)    # (batch, hidden_size)

        return self.fc(h)                      # (batch, num_classes)

# Teste rápido
model_manual = ManualRNN(MAX_VOCAB, embed_dim=64, hidden_size=128, num_classes=2)
out = model_manual(X_sample)
print(f"Saída do ManualRNN: {out.shape}  (esperado: [{BATCH}, 2])")

**Observação:** a nossa `ManualRNN` usa os *mesmos pesos* (`W_x`, `W_h`, `b`) em todos os passos de tempo — isso é o **compartilhamento de pesos** mencionado na aula.

## 3. Demonstração: Vanishing Gradient

Vamos mostrar empiricamente como o gradiente encolhe ao fluir de volta por muitos passos.

In [ ]:
def grad_norm_over_steps(T=100, hidden=64, seed=42):
    """
    Calcula a norma do gradiente de h_0 em relação a uma loss em h_T.
    Mostra como o gradiente encolhe com o número de passos T.
    """
    torch.manual_seed(seed)
    W_h = torch.randn(hidden, hidden) * 0.1  # pesos pequenos → vanishing

    h = torch.randn(hidden, requires_grad=True)
    h_t = h
    for _ in range(T):
        h_t = torch.tanh(h_t @ W_h.T)       # aplicação recorrente

    loss = h_t.sum()
    loss.backward()
    return h.grad.norm().item()

steps   = list(range(1, 101))
grads   = [grad_norm_over_steps(T=t) for t in steps]

plt.figure(figsize=(9, 4))
plt.semilogy(steps, grads, color="crimson", lw=2)
plt.xlabel("Número de passos T")
plt.ylabel("‖∂L/∂h₀‖  (escala log)")
plt.title("Vanishing Gradient na RNN vanilla")
plt.axhline(y=1e-5, color="gray", ls="--", label="limiar ≈ 0")
plt.legend()
plt.tight_layout()
plt.show()
print(f"Gradiente em T=1:   {grad_norm_over_steps(1):.4f}")
print(f"Gradiente em T=20:  {grad_norm_over_steps(20):.6f}")
print(f"Gradiente em T=100: {grad_norm_over_steps(100):.2e}")

## 4. LSTM com `nn.LSTM`

Agora vamos usar o `nn.LSTM` do PyTorch para classificação de sentimento.

In [ ]:
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim,
                 num_layers=2, num_classes=2, dropout=0.3,
                 bidirectional=False):
        super().__init__()
        self.embed   = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.lstm    = nn.LSTM(
            embed_dim, hidden_dim, num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional
        )
        D = 2 if bidirectional else 1
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_dim * D, num_classes)

    def forward(self, x):
        emb = self.embed(x)                    # (batch, seq, embed)
        _, (hn, _) = self.lstm(emb)            # hn: (layers*D, batch, hidden)
        # Usar o último estado oculto da última camada
        last = hn[-1]                          # (batch, hidden)
        return self.fc(self.dropout(last))

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, n = 0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
        loss   = criterion(logits, y)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # gradient clipping!
        optimizer.step()
        total_loss += loss.item() * len(y)
        correct    += (logits.argmax(1) == y).sum().item()
        n          += len(y)
    return total_loss / n, correct / n

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, n = 0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
        loss   = criterion(logits, y)
        total_loss += loss.item() * len(y)
        correct    += (logits.argmax(1) == y).sum().item()
        n          += len(y)
    return total_loss / n, correct / n

def run_experiment(model, epochs=5, lr=1e-3):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    history   = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    for ep in range(1, epochs + 1):
        t0 = time.time()
        tl, ta = train_epoch(model, train_loader, optimizer, criterion)
        vl, va = evaluate(model, test_loader, criterion)
        history["train_loss"].append(tl); history["train_acc"].append(ta)
        history["val_loss"].append(vl);   history["val_acc"].append(va)
        print(f"Época {ep:2d} | loss treino {tl:.4f} acc {ta:.3f} | "
              f"loss val {vl:.4f} acc {va:.3f} | {time.time()-t0:.1f}s")
    return history

In [ ]:
EPOCHS = 5

# Treinar LSTM
print("=" * 60)
print("LSTM (2 camadas, bidirecional=False)")
print("=" * 60)
lstm_model = SentimentLSTM(
    vocab_size=MAX_VOCAB, embed_dim=128, hidden_dim=256,
    num_layers=2, dropout=0.3, bidirectional=False
)
print(f"Parâmetros: {sum(p.numel() for p in lstm_model.parameters()):,}")
hist_lstm = run_experiment(lstm_model, epochs=EPOCHS)

## 5. GRU — Arquitetura Simplificada

O GRU usa apenas 2 gates (reset e update) e um único vetor de estado. Treinamos com a mesma configuração para comparar.

In [ ]:
class SentimentGRU(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim,
                 num_layers=2, num_classes=2, dropout=0.3,
                 bidirectional=False):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.gru   = nn.GRU(
            embed_dim, hidden_dim, num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional
        )
        D = 2 if bidirectional else 1
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_dim * D, num_classes)

    def forward(self, x):
        emb = self.embed(x)
        _, hn = self.gru(emb)      # GRU não tem c — só hn
        last  = hn[-1]
        return self.fc(self.dropout(last))

print("=" * 60)
print("GRU (2 camadas, bidirecional=False)")
print("=" * 60)
gru_model = SentimentGRU(
    vocab_size=MAX_VOCAB, embed_dim=128, hidden_dim=256,
    num_layers=2, dropout=0.3, bidirectional=False
)
print(f"Parâmetros: {sum(p.numel() for p in gru_model.parameters()):,}")
hist_gru = run_experiment(gru_model, epochs=EPOCHS)

## 6. LSTM Bidirecional

Com `bidirectional=True`, a rede processa a sequência em ambas as direções e concatena os estados:

$$\mathbf{h}_t = [\overrightarrow{\mathbf{h}}_t \;; \overleftarrow{\mathbf{h}}_t]$$

⚠️ Só faz sentido quando a sequência inteira está disponível de antemão (não para geração autorregressiva).

In [ ]:
print("=" * 60)
print("LSTM Bidirecional (2 camadas)")
print("=" * 60)
bilstm_model = SentimentLSTM(
    vocab_size=MAX_VOCAB, embed_dim=128, hidden_dim=128,  # hidden_dim menor: ×2 por ser bi
    num_layers=2, dropout=0.3, bidirectional=True
)
print(f"Parâmetros: {sum(p.numel() for p in bilstm_model.parameters()):,}")
hist_bilstm = run_experiment(bilstm_model, epochs=EPOCHS)

## 7. Comparativo — Curvas de Aprendizado

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

models_hist = {
    "LSTM":          hist_lstm,
    "GRU":           hist_gru,
    "BiLSTM":        hist_bilstm,
}
colors = ["steelblue", "seagreen", "darkorange"]

for ax, metric, title in zip(
    axes,
    ["val_loss", "val_acc"],
    ["Loss (validação)", "Acurácia (validação)"]
):
    for (name, hist), color in zip(models_hist.items(), colors):
        ax.plot(range(1, EPOCHS+1), hist[metric], label=name, color=color, lw=2, marker="o")
    ax.set_title(title)
    ax.set_xlabel("Época")
    ax.legend()

plt.tight_layout()
plt.show()

# Tabela resumo
print(f"{'Modelo':<12} {'Val acc':>8} {'Params':>12}")
print("-" * 34)
for (name, hist), model in zip(
    models_hist.items(),
    [lstm_model, gru_model, bilstm_model]
):
    best_acc = max(hist["val_acc"])
    params   = sum(p.numel() for p in model.parameters())
    print(f"{name:<12} {best_acc:>8.3f} {params:>12,}")

## 8. Predição em Textos Novos

In [ ]:
def predict(model, text, top_k=1):
    """Classifica um texto como positivo ou negativo."""
    model.eval()
    x = torch.tensor([encode(text)], dtype=torch.long).to(device)
    with torch.no_grad():
        logits = model(x)
        probs  = torch.softmax(logits, dim=-1)[0]
    labels = ["Negativo", "Positivo"]
    print(f"Texto : {text[:80]}...")
    print(f"Predição: {labels[probs.argmax()]}  "
          f"(neg={probs[0]:.2%}, pos={probs[1]:.2%})")
    print()

examples = [
    "This movie was absolutely fantastic! The acting was superb and the story was gripping.",
    "Terrible film. Boring plot, bad acting, complete waste of time.",
    "It was okay, nothing special but not terrible either.",
]

print("--- LSTM ---")
for t in examples:
    predict(lstm_model, t)